# MURA Training Analysis & Visualization

This notebook provides comprehensive analysis and visualization of your training results.

**What you'll see:**
- Training curves (loss, AUC, accuracy)
- Performance metrics
- Confusion matrix
- Predictions analysis
- Sample predictions with images

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Add project root to path
project_root = os.path.dirname(os.path.abspath(os.getcwd()))
sys.path.insert(0, project_root)

from config.config import Config
from models.simple_classifier import create_simple_model
from data.simple_dataset import get_simple_dataloaders
from training.metrics import MetricsCalculator

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports successful")

## 2. Load Training History

In [ ]:
# Load training history
history_path = '../results/simple_training/training_history.csv'

if os.path.exists(history_path):
    history_df = pd.read_csv(history_path)
    print("✓ Training history loaded")
    print(f"  Epochs trained: {len(history_df)}")
    print("\nHistory columns:", list(history_df.columns))
    display(history_df.head())
else:
    print(f"⚠️ History file not found at {history_path}")
    print("Please run training first: python experiments/train_simple_enhanced.py")

## 3. Training Curves Visualization

In [ ]:
# Create comprehensive training plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
epochs = range(1, len(history_df) + 1)

# Loss
axes[0, 0].plot(epochs, history_df['train_loss'], 'b-o', label='Train', linewidth=2, markersize=6)
axes[0, 0].plot(epochs, history_df['val_loss'], 'r-s', label='Validation', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# AUC-ROC
axes[0, 1].plot(epochs, history_df['train_auc'], 'b-o', label='Train', linewidth=2, markersize=6)
axes[0, 1].plot(epochs, history_df['val_auc'], 'r-s', label='Validation', linewidth=2, markersize=6)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('AUC-ROC', fontsize=12)
axes[0, 1].set_title('AUC-ROC Score', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0.5, 1.0])

# Accuracy
axes[0, 2].plot(epochs, history_df['train_acc'], 'b-o', label='Train', linewidth=2, markersize=6)
axes[0, 2].plot(epochs, history_df['val_acc'], 'r-s', label='Validation', linewidth=2, markersize=6)
axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('Accuracy', fontsize=12)
axes[0, 2].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0, 2].legend(fontsize=11)
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].set_ylim([0.5, 1.0])

# Sensitivity & Specificity
axes[1, 0].plot(epochs, history_df['val_sensitivity'], 'g-^', label='Sensitivity', linewidth=2, markersize=6)
axes[1, 0].plot(epochs, history_df['val_specificity'], 'm-v', label='Specificity', linewidth=2, markersize=6)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Score', fontsize=12)
axes[1, 0].set_title('Sensitivity & Specificity', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0.5, 1.0])

# Overfitting analysis (train vs val AUC gap)
auc_gap = history_df['train_auc'] - history_df['val_auc']
axes[1, 1].plot(epochs, auc_gap, 'purple', linewidth=2, marker='o', markersize=6)
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('AUC Gap (Train - Val)', fontsize=12)
axes[1, 1].set_title('Overfitting Analysis', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

# Summary statistics
axes[1, 2].axis('off')
summary_text = f"""Training Summary

Total Epochs: {len(history_df)}

Best Validation Metrics:
  AUC:         {history_df['val_auc'].max():.4f}
  Accuracy:    {history_df['val_acc'].max():.4f}
  Sensitivity: {history_df['val_sensitivity'].max():.4f}
  Specificity: {history_df['val_specificity'].max():.4f}

Final Epoch:
  Val AUC:     {history_df['val_auc'].iloc[-1]:.4f}
  Val Acc:     {history_df['val_acc'].iloc[-1]:.4f}

Overfitting:
  AUC Gap:     {auc_gap.iloc[-1]:.4f}
"""
axes[1, 2].text(0.1, 0.5, summary_text, fontsize=12, family='monospace',
                verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('MURA Training Analysis', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../results/simple_training/detailed_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training curves plotted")

## 4. Load Model and Evaluate

In [ ]:
# Load configuration
config = Config()

# Load model
model = create_simple_model(device='cpu', pretrained=False)
checkpoint = torch.load('../checkpoints/simple_best.pth', map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✓ Model loaded from checkpoint")
print(f"  Epoch: {checkpoint['epoch']}")
if 'metrics' in checkpoint:
    print(f"  Validation AUC: {checkpoint['metrics']['auc']:.4f}")

In [ ]:
# Load validation data
_, val_loader = get_simple_dataloaders(config, train_limit=1000, val_limit=200)

print(f"✓ Validation data loaded: {len(val_loader.dataset)} samples")

In [ ]:
# Run evaluation
from tqdm.notebook import tqdm

metrics_calc = MetricsCalculator()
all_predictions = []
all_probabilities = []
all_labels = []
all_study_paths = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        images = batch['images']
        mask = batch['mask']
        labels = batch['label']
        study_paths = batch['study_path']
        
        logits, _ = model(images, mask)
        probs = torch.sigmoid(logits).squeeze().numpy()
        preds = (probs > 0.5).astype(int)
        
        metrics_calc.update(logits, labels)
        
        all_predictions.extend(preds if isinstance(preds, np.ndarray) else [preds])
        all_probabilities.extend(probs if isinstance(probs, np.ndarray) else [probs])
        all_labels.extend(labels.numpy())
        all_study_paths.extend(study_paths)

final_metrics = metrics_calc.compute()

print("\n" + "="*60)
print("Final Evaluation Results")
print("="*60)
print(f"AUC-ROC:      {final_metrics['auc']:.4f}")
print(f"Accuracy:     {final_metrics['accuracy']:.4f}")
print(f"Sensitivity:  {final_metrics['sensitivity']:.4f}")
print(f"Specificity:  {final_metrics['specificity']:.4f}")
print(f"Precision:    {final_metrics['precision']:.4f}")
print(f"F1 Score:     {final_metrics['f1']:.4f}")
print("="*60)

## 5. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

# Create confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Normal', 'Abnormal'],
            yticklabels=['Normal', 'Abnormal'],
            annot_kws={'size': 16})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)

# Add metrics text
metrics_text = f"""Metrics:
Accuracy: {final_metrics['accuracy']:.3f}
Sensitivity: {final_metrics['sensitivity']:.3f}
Specificity: {final_metrics['specificity']:.3f}
"""
plt.text(2.3, 0.5, metrics_text, fontsize=11, family='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.tight_layout()
plt.savefig('../results/simple_training/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {cm[0,0]}")
print(f"  False Positives: {cm[0,1]}")
print(f"  False Negatives: {cm[1,0]}")
print(f"  True Positives:  {cm[1,1]}")

## 6. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(all_labels, all_probabilities)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig('../results/simple_training/roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")

## 7. Prediction Distribution

In [ ]:
# Plot probability distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Separate probabilities by true label
normal_probs = [p for p, l in zip(all_probabilities, all_labels) if l == 0]
abnormal_probs = [p for p, l in zip(all_probabilities, all_labels) if l == 1]

# Histogram
axes[0].hist(normal_probs, bins=20, alpha=0.6, label='Normal (True)', color='blue', edgecolor='black')
axes[0].hist(abnormal_probs, bins=20, alpha=0.6, label='Abnormal (True)', color='red', edgecolor='black')
axes[0].axvline(x=0.5, color='green', linestyle='--', linewidth=2, label='Threshold (0.5)')
axes[0].set_xlabel('Predicted Probability', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Prediction Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Box plot
data_to_plot = [normal_probs, abnormal_probs]
bp = axes[1].boxplot(data_to_plot, labels=['Normal', 'Abnormal'],
                      patch_artist=True, widths=0.6)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
axes[1].axhline(y=0.5, color='green', linestyle='--', linewidth=2, label='Threshold')
axes[1].set_ylabel('Predicted Probability', fontsize=12)
axes[1].set_title('Prediction Distribution by Class', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/simple_training/prediction_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPrediction Statistics:")
print(f"  Normal cases - Mean prob: {np.mean(normal_probs):.3f}, Std: {np.std(normal_probs):.3f}")
print(f"  Abnormal cases - Mean prob: {np.mean(abnormal_probs):.3f}, Std: {np.std(abnormal_probs):.3f}")

## 8. Error Analysis

In [ ]:
# Create predictions dataframe
predictions_df = pd.DataFrame({
    'study_path': all_study_paths,
    'true_label': all_labels,
    'predicted_label': all_predictions,
    'probability': all_probabilities
})

# Add prediction status
predictions_df['correct'] = predictions_df['true_label'] == predictions_df['predicted_label']
predictions_df['error_type'] = 'Correct'
predictions_df.loc[(predictions_df['true_label'] == 0) & (predictions_df['predicted_label'] == 1), 'error_type'] = 'False Positive'
predictions_df.loc[(predictions_df['true_label'] == 1) & (predictions_df['predicted_label'] == 0), 'error_type'] = 'False Negative'

# Save predictions
predictions_df.to_csv('../results/simple_training/predictions.csv', index=False)
print("✓ Predictions saved to CSV")

# Error summary
print("\nError Analysis:")
print(predictions_df['error_type'].value_counts())
print(f"\nAccuracy: {predictions_df['correct'].sum() / len(predictions_df) * 100:.2f}%")

# Show worst predictions (most confident errors)
print("\nMost Confident Errors:")
errors = predictions_df[~predictions_df['correct']].copy()
errors['confidence'] = errors['probability'].apply(lambda x: max(x, 1-x))
worst_errors = errors.nlargest(5, 'confidence')
display(worst_errors[['study_path', 'true_label', 'predicted_label', 'probability', 'error_type']])

## 9. Performance by Confidence

In [ ]:
# Analyze performance at different confidence levels
predictions_df['confidence'] = predictions_df['probability'].apply(lambda x: max(x, 1-x))

# Bin by confidence
bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
predictions_df['confidence_bin'] = pd.cut(predictions_df['confidence'], bins=bins)

# Calculate accuracy per bin
confidence_acc = predictions_df.groupby('confidence_bin')['correct'].agg(['mean', 'count'])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy by confidence
bin_centers = [0.55, 0.65, 0.75, 0.85, 0.95]
axes[0].bar(range(len(confidence_acc)), confidence_acc['mean'], color='steelblue', edgecolor='black')
axes[0].set_xticks(range(len(confidence_acc)))
axes[0].set_xticklabels(confidence_acc.index.astype(str), rotation=45)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_xlabel('Confidence Range', fontsize=12)
axes[0].set_title('Accuracy by Confidence Level', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# Sample count per bin
axes[1].bar(range(len(confidence_acc)), confidence_acc['count'], color='coral', edgecolor='black')
axes[1].set_xticks(range(len(confidence_acc)))
axes[1].set_xticklabels(confidence_acc.index.astype(str), rotation=45)
axes[1].set_ylabel('Number of Samples', fontsize=12)
axes[1].set_xlabel('Confidence Range', fontsize=12)
axes[1].set_title('Sample Distribution by Confidence', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/simple_training/confidence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Accuracy by Confidence Level:")
display(confidence_acc)

## 10. Summary Report

In [ ]:
# Generate comprehensive summary
print("="*70)
print(" "*20 + "MURA TRAINING SUMMARY")
print("="*70)

print("\n📊 TRAINING CONFIGURATION")
print("-"*70)
if 'args' in checkpoint:
    args = checkpoint['args']
    print(f"  Epochs:           {args.get('epochs', 'N/A')}")
    print(f"  Batch Size:       {args.get('batch_size', 'N/A')}")
    print(f"  Learning Rate:    {args.get('lr', 'N/A')}")
    print(f"  Training Samples: {args.get('train_samples', 'N/A')}")
    print(f"  Val Samples:      {args.get('val_samples', 'N/A')}")

print("\n🎯 FINAL PERFORMANCE")
print("-"*70)
print(f"  AUC-ROC:          {final_metrics['auc']:.4f}")
print(f"  Accuracy:         {final_metrics['accuracy']:.4f}")
print(f"  Sensitivity:      {final_metrics['sensitivity']:.4f}")
print(f"  Specificity:      {final_metrics['specificity']:.4f}")
print(f"  Precision:        {final_metrics['precision']:.4f}")
print(f"  F1 Score:         {final_metrics['f1']:.4f}")

print("\n📈 TRAINING PROGRESS")
print("-"*70)
print(f"  Best Val AUC:     {history_df['val_auc'].max():.4f} (Epoch {history_df['val_auc'].idxmax() + 1})")
print(f"  Final Val AUC:    {history_df['val_auc'].iloc[-1]:.4f}")
print(f"  AUC Improvement:  {(history_df['val_auc'].iloc[-1] - history_df['val_auc'].iloc[0]):.4f}")

print("\n🔍 CONFUSION MATRIX")
print("-"*70)
print(f"  True Positives:   {cm[1,1]}")
print(f"  True Negatives:   {cm[0,0]}")
print(f"  False Positives:  {cm[0,1]}")
print(f"  False Negatives:  {cm[1,0]}")

print("\n💾 SAVED FILES")
print("-"*70)
print(f"  Model:            checkpoints/simple_best.pth")
print(f"  History CSV:      results/simple_training/training_history.csv")
print(f"  Predictions:      results/simple_training/predictions.csv")
print(f"  Training Plots:   results/simple_training/*.png")

print("\n" + "="*70)
print("✓ Analysis Complete!")
print("="*70)

## 📌 Key Takeaways

Based on this analysis:

1. **Model Performance:** Check if AUC ≥ 0.75 (good for subset training)
2. **Overfitting:** Look at train vs validation gap in AUC
3. **Confidence:** Higher confidence predictions should be more accurate
4. **Balance:** Check sensitivity vs specificity trade-off

### Next Steps:
- If AUC < 0.75: Train longer or with more data
- If overfitting: Add regularization or reduce model complexity  
- If good results: Scale up to full dataset and GPU
- For publication: Use full ViT model on complete dataset